# Gomoku Small Version

Board:  4 x 4

Win: 3 in a row (horizontaly or vertically), 4 diagonally

In [8]:
blue = '\033[94m'
pink = '\033[95m'
reset = '\033[0m'
print("\033[94m Blue Color")
print("\033[97m Grey Color")
print("\033[95m Pink Color")

 Blue Color
 Grey Color
 Pink Color


In [9]:
class CurrentBoard():
    def __init__(self, board=None):
        if board is None:
            self.setup_of_board = " " * 16
            self.board = self.setup_of_board
        else:
            self.board = board
        self.state = self.state_of_play()
    
    def colored(self, char):
      if char == "B":
          return blue + "O" + reset
      if char == "P":
          return pink + "O" + reset
      return char

    def display_board(self, game_display=False):
        c = []
        if game_display:
            index_display = 0
            for i in range(len(self.board)):
                if self.board[i] == " ":
                    c.append(str(index_display))
                    index_display += 1
                else:
                    c.append(self.colored(self.board[i]))
        else:
            c = [self.colored(ch) for ch in self.board]

        print(c[0]  + '---' + c[1]  + '---' + c[2]  + '---' + c[3])
        print('|   |   |   |')
        print(c[4]  + '---' + c[5]  + '---' + c[6]  + '---' + c[7])
        print('|   |   |   |')
        print(c[8]  + '---' + c[9]  + '---' + c[10] + '---' + c[11])
        print('|   |   |   |')
        print(c[12] + '---' + c[13] + '---' + c[14] + '---' + c[15])

    def all_possible_moves(self, forPlayer):
      all_moves = []
      for i in range(len(self.board)):
        if self.board[i] == " ":
          string_for_board = self.board[:i] + forPlayer + self.board[i+1:]
          all_moves.append(CurrentBoard(board=string_for_board))
      return all_moves
    
    def other_player(self, forPlayer):
      if forPlayer == "B":
        return "P"
      return "B"

    def Eq3(self, i1,i2,i3):
      a = self.board[i1]
      b = self.board[i2]
      c = self.board[i3]
      return (a==b) and (b==c) and (a!=" ")
    
    def Eq4(self, i1,i2,i3,i4):
      a = self.board[i1]
      b = self.board[i2]
      c = self.board[i3]
      d = self.board[i4]
      return (a==b) and (b==c) and (c==d) and (a!=" ")
    
    def state_of_play(self):
    # "B" if Blue wins
    # "G" if Gray wins
    # "D" if draw
    # "U" if unfinished

    # Board layout (4x4):
    # 00 01 02 03
    # 04 05 06 07
    # 08 09 10 11
    # 12 13 14 15

    # Check rows
        if self.Eq3(0,1,2) or self.Eq3(1,2,3):
            return self.board[1]
        if self.Eq3(4,5,6) or self.Eq3(5,6,7):
            return self.board[5]
        if self.Eq3(8,9,10) or self.Eq3(9,10,11):
            return self.board[9]
        if self.Eq3(12,13,14) or self.Eq3(13,14,15):
            return self.board[13]

    # Check columns
        if self.Eq3(0,4,8) or self.Eq3(4,8,12):
            return self.board[4]
        if self.Eq3(1,5,9) or self.Eq3(5,9,13):
            return self.board[5]
        if self.Eq3(2,6,10) or self.Eq3(6,10,14):
            return self.board[6]
        if self.Eq3(3,7,11) or self.Eq3(7,11,15):
            return self.board[7]

    # Check diagonals
        if self.Eq4(0,5,10,15):
            return self.board[5]
        if self.Eq4(3,6,9,12):
            return self.board[6]

    # If no winner is found
        if " " in self.board:
            return "U"
        return "D"

    def evaluation(self, forPlayer):
        opponent = self.other_player(forPlayer)

        score = 0

        all_win_lines = [
            [0,1,2], [1,2,3],       
            [4,5,6], [5,6,7],       
            [8,9,10],[9,10,11],     
            [12,13,14],[13,14,15],  
            [0,4,8], [4,8,12],      
            [1,5,9], [5,9,13],      
            [2,6,10],[6,10,14],     
            [3,7,11],[7,11,15],     
            [0,5,10,15],            
            [3,6,9,12]              
        ]

        for line in all_win_lines:
            ai_count = 0
            opp_count = 0

            for i in line:
                if self.board[i] == forPlayer:
                    ai_count += 1

                if self.board[i] == opponent:
                    opp_count += 1

            if opp_count == 0:
                if ai_count > 0:
                    score = score + ai_count * ai_count

            if ai_count == 0:
                if opp_count > 0:
                    score = score - opp_count * opp_count

        centre = [5, 6, 9, 10]

        for pos in centre:
            if self.board[pos] == forPlayer:
                score = score + 1
            if self.board[pos] == opponent:
                score = score - 1

        return score

In [10]:
board = CurrentBoard()
board.display_board(game_display=True)

0---1---2---3
|   |   |   |
4---5---6---7
|   |   |   |
8---9---10---11
|   |   |   |
12---13---14---15


In [11]:
class SearchTreeNode():
    def __init__(self, current_board, forPlayer, ply=0, ai_player=None):
        self.board = current_board
        self.ply_depth = ply
        self.value_is_assigned = False
        self.children = []
        self.max_depth = 6
        
        if ai_player is not None:
            self.ai_player = ai_player
        else:
            self.ai_player = forPlayer

        if current_board.state == "U" and self.ply_depth < self.max_depth:
            moves = current_board.all_possible_moves(forPlayer)
            moves.sort(key=lambda b: b.evaluation(forPlayer), reverse=True)
            
            for move in moves:
                self.children.append(SearchTreeNode(move, self.board.other_player(forPlayer), self.ply_depth + 1, self.ai_player))

        else:
            self.value_is_assigned = True
            if self.board.state == "D":
                self.value = 0
            elif self.board.state == self.ai_player:
                self.value = 1000
            elif self.board.state == self.board.other_player(self.ai_player):
                self.value = -1000
            else:
                self.value = self.board.evaluation(self.ai_player)


    def min_max_value(self):
        if self.value_is_assigned:
            return self.value

        self.children = sorted(self.children, key=lambda x: x.min_max_value())

        self.value_is_assigned = True

        if self.ply_depth % 2 == 0:
            self.value = self.children[-1].value
        else:
            self.value = self.children[0].value

        return self.value

In [12]:
def replace_space(board_string, index, charBorG):
  space_ind = 0
  board_index = 0
  for c in board_string:
    if c == " ":
      if space_ind == index:
        return board_string[:board_index] + charBorG + board_string[board_index+1:]
      space_ind += 1
    board_index += 1
  return board_string

In [13]:
def play_game():
  response = input(" Do you want to play as Blue or Pink?")
  player_playing_as = response
  AI_playing_as = CurrentBoard().other_player(player_playing_as)
  if(player_playing_as == "B"): player_playing_first = True 
  else: player_playing_first = False
  player_turn = player_playing_first
  cb = CurrentBoard()

  while True:
    if player_turn:
      # display current board for player
      cb.display_board(game_display=True)
      player_move = input("Make your choice ")
      player_move = int(player_move)
      boards_string = replace_space(cb.board, player_move, player_playing_as)
      cb = CurrentBoard(boards_string)

    else:
      # AI move
      print(" AI is making the move for the following board, playing as " + AI_playing_as)
      cb.display_board()
      print("Constructing the search tree......")
      st = SearchTreeNode(cb, AI_playing_as)
      print("Applying the min Max algorithm")
      st.min_max_value()
      print(" Choosing the best move")
      
      if AI_playing_as == "B":
        cb = max(st.children, key=lambda x: x.value).board
      else:
        cb = min(st.children, key=lambda x: x.value).board

    # Check for end game
    if cb.state != "U":
      cb.display_board()
    if cb.state == "D":
      print("Game is a draw, I can't believe it, you got lucky")
      return
    if cb.state == AI_playing_as:
      print(" I win too easy!!!!")
      return
    if cb.state == player_playing_as:
      print("This is just not possible")
      return
    # Toggle AI and Player move
    player_turn = not player_turn


In [14]:
play_game()

 AI is making the move for the following board, playing as B
 --- --- --- 
|   |   |   |
 --- --- --- 
|   |   |   |
 --- --- --- 
|   |   |   |
 --- --- --- 
Constructing the search tree......
Applying the min Max algorithm
 Choosing the best move
0---1---2---3
|   |   |   |
4---O---5---6
|   |   |   |
7---8---9---10
|   |   |   |
11---12---13---14
 AI is making the move for the following board, playing as B
 --- --- --- 
|   |   |   |
 ---O--- --- 
|   |   |   |
 ---O--- --- 
|   |   |   |
 --- --- --- 
Constructing the search tree......
Applying the min Max algorithm
 Choosing the best move
0---1---2---3
|   |   |   |
4---O---O---5
|   |   |   |
6---O---7---8
|   |   |   |
9---10---11---12
 AI is making the move for the following board, playing as B
 --- --- --- 
|   |   |   |
 ---O---O---O
|   |   |   |
 ---O--- --- 
|   |   |   |
 --- --- --- 
Constructing the search tree......
Applying the min Max algorithm
 Choosing the best move
0---1---2---3
|   |   |   |
4---O---O---O
|   |  